# 01 · Combine source files → single raw file

Merges the 7 raw files into `raw_combined.csv`, keeping every source column except derived
ones (recomputed in `03_enrich`).

**Multi-recipient awards:** one award can name several OIs, recorded as one row per OI sharing
a `GRANT_ID`. These rows are legitimate and kept distinct. Identity has two keys:
- `source_grantid` = `source::GRANT_ID` — award-level key the duplicates table uses.
- `record_uid` = `source::GRANT_ID::OI` — row-unique (one OI per row).

Blank `GRANT_ID`s (990-sourced) get a deterministic `SYN-` hash; any residual exact-duplicate
`record_uid` gets a `#n` suffix as a final uniqueness guard.

In [1]:
import pandas as pd
from pathlib import Path
import hashlib, re

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## CONFIG

In [6]:
# from google.colab import drive; drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding/Data from COKI_FINAL_2026-06-08')

SOURCE_FILES = {
    'manual':   'results_manual20260610.csv',
    'DOD':      'results_dod_20260521.csv',
    'HHS':      'results_hhs_20260521.csv',
    'NASA':     'results_nasa_20260521.csv',
    'scrapers': 'results_scrapers20260513.csv',
    'openaire': 'results_openaire20260608.csv',
    'openalex': 'results_openalex20260526.csv',
}
# Working file lands in the parent (Grant Funding/), leaving the raw COKI export untouched.
OUTPUT_FILE = BASE_DIR.parent / 'raw_combined.csv'

ID_COLUMN = 'GRANT_ID'
ID_COLUMN_BY_SOURCE = {}
COL_OI = 'OI'
HASH_FIELDS = ['FUNDER','RECIPIENT','OI','GRANT_YEAR','AMOUNT']

# Derived columns stripped from raw (matched by prefix, so naming variants are all caught):
#   AMOUNT_USD, DASH_AMT, FUNDER_ROR, RECIPIENT_ROR -> recomputed in 03_enrich
#   IP_SOLNCAT* / OI_SOLN_CAT*                       -> derived from the OI match (downstream)
#   GRANT_CAT*                                       -> output of the award-type classifier (downstream)
DROP_PREFIXES = ['AMOUNT_USD','DASH_AMT','FUNDER_ROR','RECIPIENT_ROR',
                 'IP_SOLNCAT','OI_SOLN_CAT','GRANT_CAT']

# Combined-OI strings that name several OIs in one row -> split into one row per OI, so the
# award is correctly treated as multi-recipient. Keys matched case-insensitively.
SPLIT_OI = {'biorxiv/medrxiv': ['bioRxiv', 'medRxiv']}

## Load, key (award + row), combine

In [7]:
def read_any(p): return pd.read_excel(p,dtype=str) if p.suffix.lower() in {'.xlsx','.xls'} else pd.read_csv(p,dtype=str)
def norm_oi(v): return re.sub(r'\s+',' ',str(v or '').strip().lower())

def base_grantid(df, source):
    col = ID_COLUMN_BY_SOURCE.get(source, ID_COLUMN)
    gid = df[col].astype(str).str.strip() if col in df.columns else pd.Series(['']*len(df), index=df.index)
    blank = gid.isin(['','nan','None','NaN']) | gid.isna()
    if blank.any():
        def h(row): return 'SYN-'+hashlib.md5((source+'|'+'|'.join(str(row.get(f,'')).strip().lower() for f in HASH_FIELDS)).encode()).hexdigest()[:12]
        hashed = df.apply(h, axis=1)
        occ = hashed.groupby(hashed).cumcount()
        hashed = hashed.where(occ==0, hashed+'-'+occ.astype(str))
        gid = gid.mask(blank, hashed)
        print(f'  ! {source}: {int(blank.sum())} blank ids hashed')
    return gid

def split_combined_oi(df, source):
    if COL_OI not in df.columns or not SPLIT_OI: return df
    key=df[COL_OI].map(norm_oi); mask=key.isin(SPLIT_OI)
    if not mask.any(): return df
    parts=[]
    for _,row in df[mask].iterrows():
        for sub in SPLIT_OI[norm_oi(row[COL_OI])]:
            r=row.copy(); r[COL_OI]=sub; parts.append(r)
    print(f'  ! {source}: split {int(mask.sum())} combined-OI row(s) into {len(parts)} OI rows')
    return pd.concat([df[~mask], pd.DataFrame(parts)], ignore_index=True)

frames=[]
for source,fname in SOURCE_FILES.items():
    df = read_any(BASE_DIR/fname)
    df['__gid'] = base_grantid(df, source).values      # mint award id on the ORIGINAL row
    df = split_combined_oi(df, source)                 # split AFTER id so sub-rows share the award
    oi = df[COL_OI].map(norm_oi) if COL_OI in df.columns else pd.Series(['']*len(df), index=df.index)
    df.insert(0,'source_dataset', source)
    df.insert(1,'source_grantid', source+'::'+df['__gid'].astype(str))
    df.insert(2,'record_uid', source+'::'+df['__gid'].astype(str)+'::'+oi)
    df=df.drop(columns='__gid')
    print(f'{source:10s} {len(df):>8,} rows  {len(df.columns):>3} cols  <- {fname}')
    frames.append(df)

raw = pd.concat(frames, ignore_index=True, sort=False)
dropped=[c for c in raw.columns if any(c.startswith(p) for p in DROP_PREFIXES)]; raw=raw.drop(columns=dropped)
print('\nStripped derived columns:', dropped or '(none)')
if 'AMOUNT' in raw.columns: raw['AMOUNT']=pd.to_numeric(raw['AMOUNT'],errors='coerce')

  ! manual: 24 blank ids hashed
  ! manual: split 1 combined-OI row(s) into 2 OI rows
manual           25 rows   27 cols  <- results_manual20260610.csv
DOD              18 rows   27 cols  <- results_dod_20260521.csv
HHS             198 rows   27 cols  <- results_hhs_20260521.csv
NASA             24 rows   27 cols  <- results_nasa_20260521.csv
scrapers        864 rows   27 cols  <- results_scrapers20260513.csv
openaire        941 rows   28 cols  <- results_openaire20260608.csv
openalex      2,486 rows   27 cols  <- results_openalex20260526.csv

Stripped derived columns: ['FUNDER_ROR', 'RECIPIENT_ROR', 'AMOUNT_USD', 'IP_SOLNCAT_1', 'IP_SOLNCAT_2', 'GRANT_CAT_1', 'GRANT_CAT_2', 'GRANT_CAT']


## Final uniqueness guard + write

A repeated `record_uid` now means a genuine exact-duplicate row (same award, same OI). Suffix
with `#n` so nothing is silently lost, and report the count for review.

In [8]:
occ = raw.groupby('record_uid').cumcount()
n_exact = int((occ>0).sum())
raw['record_uid'] = raw['record_uid'].where(occ==0, raw['record_uid']+'#'+occ.astype(str))
print('Exact-duplicate rows suffixed (review if >0):', n_exact)
print(f'Combined rows: {len(raw):,} | columns: {len(raw.columns)} | duplicate record_uids: {int(raw.record_uid.duplicated().sum())}')
raw.to_csv(OUTPUT_FILE, index=False); print('Wrote', OUTPUT_FILE)

Exact-duplicate rows suffixed (review if >0): 10
Combined rows: 4,556 | columns: 20 | duplicate record_uids: 0
Wrote /content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding/raw_combined.csv
